# Travel Personas
- using k means to try and find the type of traveller i am on different days

### Imports

In [1]:
import pandas as pd
import numpy as np
import scipy as sp
import plotly.express as px
import plotly.graph_objects as go
import datetime

import json

### Variables

In [17]:
hours_in_a_day = 24

In [2]:
regions = {
    'South America': ['Colombia','Ecuador','Peru','Bolivia','Brasil'],
    'Central America': ['Guatemala','Honduras','Nicaragua','El Salvador'],
    'Southern Africa': ['South Africa','Malawi','Mozambique','Eswatini','Lesotho']
}

In [9]:
base_currency = ['CAD']

In [10]:
countries = {
    'Colombia' : {
        'color': '#EFCA08',
        'currency': ['COP']
    },
    'Ecuador' : {
        'color': '#FF9000',
        'currency': ['USD']
    },
    'Peru' : {
        'color': '#DB5461',
        'currency': ['PEN']
    },
    'Bolivia' : {
        'color': '#A4243B',
        'currency': ['BOB']
    },
    'Brasil' : {
        'color': '#94BFA7',
        'currency': ['BRL']
    },
    'Guatemala' : {
        'color': '#86A5D9',
        'currency': ['GTQ','USD']
    },
    'Honduras' : {
        'color': '#0B3954',
        'currency': ['HNL','USD']
    },
    'Nicaragua' : {
        'color': '#57C4E5',
        'currency': ['NIO','USD']
    },
    'El Salvador': {
        'color': '#273043',
        'currency': ['USD']
    },
    'South Africa': {
        'color': '#D4E09B',
        'currency': ['ZAR']
    },
    'Malawi': {
        'color': '#D72638',
        'currency': ['MK','BMK']
    },
    'Mozambique': {
        'color': '#2D5C1E',
        'currency': ['MZN']
    },
    'Eswatini': {
        'color': '#AFA3B8',
        'currency': ['ZAR']
    },
    'Lesotho': {
        'color': '#8EF9F3',
        'currency': ['ZAR']
    }
}

In [83]:
transport_types = {
    'hitchhiking': {
        'values': ['hitchhiking'],
        'mapping': 0
    },
    'self': {
        'values': ['car', 'car share'],
        'mapping': 1
    },
    'bus': {
        'values': ['bus'],
        'mapping': 2
    },
    'tourist': {
        'values': ['van', 'truck', 'flight', 'teleferico', 'jeep', 'boat', 'shuttle'],
        'mapping': 3
    },
    'local': {
        'values': ['chicken bus', 'mini bus', 'chapa', 'kombi', 'collectivo', 'train'],
        'mapping': 4
    },
    'multiple': {
        'values': [],
        'mapping': 5
    }
}

### Functions

In [11]:
def map_countries(df, timeline):
    for idx, row in timeline.iterrows():
        # get all rows that occur in the dates within a country timeline and see if the row has the currency of a country
        clip = df[(df.date >= row.entry_date) & (df.date <= row.exit_date) & (df.currency.isin(countries[row.country]['currency'] + base_currency))]
    
        df.loc[clip.index.values, 'country'] = row.country

    return df

In [18]:
# formatting columns to be nicer to work with
def format_names(df):
    for col in df.columns:
        df = df.rename(columns={col: col.replace(' ','_')})

    return df

### Base data

In [12]:
timeline = pd.read_csv('../data/country_timeline.csv')
timeline['entry_date'] = pd.to_datetime(timeline['entry_date'])#, format='%m/%d/%Y')
timeline['exit_date'] = pd.to_datetime(timeline['exit_date'])#, format='%m/%d/%Y')

In [13]:
df = pd.read_csv('../data/money_manager_2024_2025.csv')
df.columns = df.columns.str.lower()
df = df.rename(columns={'income/expense':'income_expense',' ':'date'})

In [14]:
df['date'] = pd.to_datetime(df['date'], format='%m/%d/%Y %H:%M:%S').dt.normalize()
df['month'] = df['date'].dt.strftime("%B %Y")

In [15]:
df = map_countries(df, timeline)

In [16]:
df.head()

,date,account,category,subcategory,note,cad,income_expense,description,amount,currency,account.1,month,country
0,2025-09-25,Card,Other,NaN,Sending Rochelle's batteries,6.69,Expense,NaN,90.0,ZAR,6.69,September 2025,South Africa
1,2025-09-25,Card,🍜 Food,NaN,Panini,2.60,Expense,NaN,35.0,ZAR,2.60,September 2025,South Africa
2,2025-09-24,Card,🤑 Allowance,NaN,NaN,60.00,Income,NaN,60.0,CAD,60.00,September 2025,South Africa
3,2025-09-24,Cash,🎁 Gift,NaN,Handful of beaded animals,19.32,Expense,NaN,260.0,ZAR,19.32,September 2025,South Africa
4,2025-09-23,Card,🤑 Allowance,NaN,NaN,60.00,Income,NaN,60.0,CAD,60.00,September 2025,South Africa


In [50]:
transport = pd.read_csv('../data/transport_data.csv')
transport = format_names(transport)
transport['date'] = pd.to_datetime(transport['date'], format='%m/%d/%Y')

# have to get rid of the random extra column i added for some reason
transport = transport.drop('Unnamed:_8', axis=1)

# fixing hitch hiking split
transport['type'] = transport['type'].str.replace('hitch hiking','hitchhiking')
transport['type'] = transport['type'].str.replace('bus ','bus')

transport.head()

,city,destination,region,country,date,approximate_time,type,time_of_day
0,Cartagena,Santa Marta,Bolivar,Colombia,2024-03-06,4.00,collectivo,day
1,Santa Marta,Sierra Nevada Park,Magdalena,Colombia,2024-03-08,1.00,van,day
2,Sierra Nevada Park,Tayrona Park,Magdalena,Colombia,2024-03-13,0.50,van,day
3,Tayrona Park,Santa Marta,Magdalena,Colombia,2024-03-14,0.75,bus,day
4,Santa Marta,Taganga,Magdalena,Colombia,2024-03-14,0.25,collectivo,day


## Making the feature vectors
- 1 day into a bunch of features
- spending:
  - number times purchases for each category
  - sum of dollars spent
  - allowance
  - diff between allowance and spending
  - number of time purchases made
  - split type of accomodations into values: dorm, camping, suite, unpaid
- transport:
  - hours spent travelling
  - number of different transports taken
  - transport type label
- activities/tours:
  - activity done (will need to map this out to different days because activities are put in by date & length of time)
  - tour done (will need to map this out to different days because activities are put in by date & length of time)
  - label of activity type
  - label of tour done
  - time spent on activity
  - time spent on tour
- location data:
  - country numerical value

#### Spending Features

#### Transport Features
- transport type: made the category more basic, but sometimes there is more than 1 type of travel... maybe add a final group that is mixed?

In [84]:
# invert the dictionary
type_to_category = {
    v: category 
    for category, info in transport_types.items() 
    for v in info['values']
}

# apply mapping to dataframe
transport['category'] = transport['type'].map(mapping)

In [85]:
# get all of the transport categories that have happened in one day
t_agg = transport.groupby(['country','date']).agg({'approximate_time':'sum', 'type':'count', 'category':lambda x: list(set(x))}).reset_index()

# simplify the category: if multiple, replace with value 'multiple'
t_agg['category'] = t_agg['category'].apply(lambda x: x[0] if len(x) == 1 else 'multiple')

In [86]:
# extract the mapping
category_id_map = {k: v['mapping'] for k, v in transport_types.items()}

# assign the mapping
t_agg['category_id'] = t_agg['category'].map(category_id_map)

In [87]:
t_agg.head()

,country,date,approximate_time,type,category,category_id
0,Bolivia,2024-07-02,0.5,1,bus,2
1,Bolivia,2024-07-03,4.0,1,tourist,3
2,Bolivia,2024-07-06,6.5,2,multiple,5
3,Bolivia,2024-07-08,3.0,2,tourist,3
4,Bolivia,2024-07-10,13.0,1,bus,2


#### Activities/Tours Features